# Corrective RAG (CRAG) từ Scratch — Pháp Lý Việt Nam

Triển khai thủ công pipeline **Corrective RAG** cho tài liệu pháp lý Việt Nam — một pipeline truy xuất agentic thực hiện đánh giá, viết lại và xác thực ở từng bước:
1. Truy xuất tài liệu ứng viên từ vector store cục bộ
2. Đánh giá từng tài liệu về mức độ liên quan; viết lại truy vấn và thử lại nếu không có tài liệu nào đạt yêu cầu
3. Tạo câu trả lời chỉ dựa trên các tài liệu đã được xác thực
4. Kiểm tra câu trả lời cuối cùng so với các sự kiện đã truy xuất để phát hiện ảo giác

In [1]:
import os
import math
import openai
from langchain_huggingface import HuggingFaceEmbeddings

# --- Configuration Constants ---
LLM_URL = "https://api.openai.com/v1"
LLM_API_KEY = os.environ.get("LLM_API_KEY")
LLM_MODEL=os.environ.get("LLM_MODEL", "gpt-4o-mini")

# Initialize dedicated clients for individual endpoints
llm_client = openai.OpenAI(base_url=LLM_URL, api_key=LLM_API_KEY)

embeddings_engine = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## Bước 1 — Vector Store In-Memory

`TinyVectorDB` mã hóa tài liệu bằng `HuggingFaceEmbeddings` (BAAI/bge-m3) và truy xuất qua độ tương đồng cosine — một lớp tìm kiếm ngữ nghĩa nhẹ, không phụ thuộc thư viện ngoài.

In [2]:
# --- Step 1: Vector Database Powered by LangChain Embeddings ---

def cosine_similarity(v1, v2):
    dot_product = sum(x * y for x, y in zip(v1, v2))
    magnitude1 = math.sqrt(sum(x * x for x in v1))
    magnitude2 = math.sqrt(sum(x * x for x in v2))
    if not magnitude1 or not magnitude2:
        return 0
    return dot_product / (magnitude1 * magnitude2)

class TinyVectorDB:
    def __init__(self):
        self.knowledge_base = []

    def add_documents(self, docs: list[str]):
        # Batch encode vectors using LangChain wrapper
        vector_list = embeddings_engine.embed_documents(docs)
        for doc, embedding in zip(docs, vector_list):
            self.knowledge_base.append({"text": doc, "embedding": embedding})

    def query(self, query_text: str, top_k=2):
        # Encode target query vector
        query_emb = embeddings_engine.embed_query(query_text)
        results = []
        for item in self.knowledge_base:
            sim = cosine_similarity(query_emb, item["embedding"])
            results.append((item["text"], sim))
        results.sort(key=lambda x: x[1], reverse=True)
        return [text for text, sim in results[:top_k]]


## Bước 2 — Bộ Đánh Giá & Viết Lại Truy Vấn bằng LLM

Ba công cụ hỗ trợ bởi LLM điều khiển vòng lặp corrective:
- **`grade_document`** — đánh giá xem tài liệu được truy xuất có liên quan đến câu hỏi hay không
- **`rewrite_query`** — diễn đạt lại truy vấn bằng thuật ngữ pháp lý chính xác để cải thiện khả năng tìm kiếm khi không tìm thấy tài liệu liên quan
- **`grade_generation`** — kiểm tra câu trả lời cuối cùng có hoàn toàn căn cứ vào các sự kiện đã truy xuất không (bảo vệ chống ảo giác)

In [ ]:
# --- Bước 2: Các Công Cụ LLM cho Pipeline CRAG ---

def call_llm(system_prompt: str, user_prompt: str) -> str:
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0
    )
    return response.choices[0].message.content.strip().lower()

def grade_document(doc: str, query: str) -> bool:
    system_prompt = (
        "Bạn là một chuyên gia đánh giá tài liệu pháp lý Việt Nam. "
        "Nhiệm vụ của bạn là xác định xem đoạn văn bản được truy xuất có liên quan đến câu hỏi của người dùng hay không. "
        "Chỉ trả lời đúng một từ: 'yes' nếu liên quan, 'no' nếu không liên quan."
    )
    user_prompt = f"Văn bản: {doc}\nCâu hỏi: {query}\nVăn bản này có liên quan đến câu hỏi không?"
    res = call_llm(system_prompt, user_prompt)
    return "yes" in res

def rewrite_query(original_query: str) -> str:
    system_prompt = (
        "Bạn là trợ lý AI chuyên tối ưu hóa truy vấn tìm kiếm trong lĩnh vực pháp luật Việt Nam. "
        "Hãy viết lại truy vấn rõ ràng hơn bằng cách sử dụng thuật ngữ pháp lý chính xác (ví dụ: tên luật, nghị định, điều khoản). "
        "Chỉ xuất ra chuỗi truy vấn đã được cải thiện, không giải thích thêm."
    )
    user_prompt = f"Tối ưu hóa truy vấn sau để tìm kiếm văn bản pháp luật hiệu quả hơn: {original_query}"
    return call_llm(system_prompt, user_prompt)

def grade_generation(generation: str, documents: list[str]) -> bool:
    system_prompt = (
        "Bạn là chuyên gia kiểm tra tính chính xác của câu trả lời pháp lý. "
        "Đánh giá xem câu trả lời có được hỗ trợ hoàn toàn bởi các sự kiện đã cung cấp hay không — "
        "không được thêm thông tin ngoài văn bản gốc. "
        "Chỉ trả lời đúng một từ: 'yes' nếu câu trả lời căn cứ vào sự kiện, 'no' nếu có ảo giác."
    )
    docs_text = "\n".join(documents)
    user_prompt = f"Sự kiện:\n{docs_text}\n\nCâu trả lời: {generation}\nCâu trả lời có được căn cứ hoàn toàn từ các sự kiện trên không?"
    res = call_llm(system_prompt, user_prompt)
    return "yes" in res

## Bước 3 — Vòng Lặp Điều Khiển Agentic

`agentic_rag_loop` điều phối toàn bộ pipeline CRAG:
1. Truy xuất → đánh giá tài liệu → viết lại truy vấn nếu tất cả tài liệu không đạt (tối đa `max_attempts` lần)
2. Tạo câu trả lời từ ngữ cảnh đã được xác thực
3. Chạy kiểm tra ảo giác trên câu trả lời; trả về fallback an toàn nếu thất bại

In [ ]:
# --- Bước 3: Vòng Lặp Điều Khiển Agentic ---

def agentic_rag_loop(db: TinyVectorDB, query: str, max_attempts=3):
    current_query = query
    attempts = 0
    valid_docs = []

    while attempts < max_attempts:
        print(f"\n[Lần thử {attempts+1}] Tìm kiếm với BGE-M3 cho: '{current_query}'")
        retrieved_docs = db.query(current_query, top_k=2)

        valid_docs = [doc for doc in retrieved_docs if grade_document(doc, query)]

        if valid_docs:
            print(f"-> Tìm thấy {len(valid_docs)} tài liệu liên quan.")
            break
        else:
            print("-> Không tìm thấy tài liệu liên quan. Đang viết lại truy vấn...")
            current_query = rewrite_query(current_query)
            attempts += 1

    if not valid_docs:
        return "Xin lỗi, tôi không tìm thấy thông tin đáng tin cậy nào để trả lời câu hỏi của bạn."

    context = "\n".join(valid_docs)
    system_prompt = (
        "Bạn là trợ lý pháp lý chuyên về pháp luật Việt Nam. "
        "Hãy trả lời câu hỏi dựa hoàn toàn vào ngữ cảnh được cung cấp. "
        "Trả lời bằng tiếng Việt, ngắn gọn và chính xác."
    )
    user_prompt = f"Ngữ cảnh:\n{context}\n\nCâu hỏi: {query}\nTrả lời:"

    generation = call_llm(system_prompt, user_prompt)
    print(f"\n[Câu trả lời được tạo ra]: {generation}")

    is_grounded = grade_generation(generation, valid_docs)
    if is_grounded:
        print("-> Kiểm tra thành công: Câu trả lời có căn cứ từ tài liệu.")
        return generation
    else:
        print("-> Kiểm tra thất bại: Phát hiện ảo giác! Kích hoạt fallback.")
        return "Dự phòng: Câu trả lời được tạo ra không vượt qua kiểm tra an toàn."

## Bước 4 — Cơ Sở Tri Thức

Tập dữ liệu pháp lý Việt Nam mô phỏng, bao gồm các luật, nghị định và thông tư. Một số văn bản có nội dung chồng lấp hoặc dẫn chiếu chéo để kiểm tra khả năng lọc của bộ đánh giá và kích hoạt cơ chế viết lại truy vấn.

In [ ]:
db = TinyVectorDB()
mock_data = [
    # Nhóm: Luật Lao động
    "Theo Điều 105 Bộ luật Lao động 2019, thời giờ làm việc bình thường không quá 8 giờ trong một ngày và không quá 48 giờ trong một tuần.",
    "Điều 107 Bộ luật Lao động 2019 quy định người lao động làm thêm giờ không được vượt quá 50% số giờ làm việc bình thường trong ngày; trường hợp áp dụng quy định làm việc theo tuần thì tổng số giờ làm việc bình thường và số giờ làm thêm không quá 12 giờ trong một ngày.",
    "Điều 113 Bộ luật Lao động 2019 quy định người lao động làm việc đủ 12 tháng cho một người sử dụng lao động được nghỉ hằng năm hưởng nguyên lương.",

    # Nhóm: Luật Doanh nghiệp
    "Theo Điều 47 Luật Doanh nghiệp 2020, công ty trách nhiệm hữu hạn hai thành viên trở lên là doanh nghiệp có từ 02 đến 50 thành viên là tổ chức, cá nhân.",
    "Điều 111 Luật Doanh nghiệp 2020 quy định công ty cổ phần là doanh nghiệp có vốn điều lệ được chia thành nhiều phần bằng nhau gọi là cổ phần; số lượng cổ đông tối thiểu là 03 và không hạn chế số lượng tối đa.",
    "Nghị định 01/2021/NĐ-CP ngày 04/01/2021 của Chính phủ quy định về đăng ký doanh nghiệp, bao gồm hồ sơ, trình tự, thủ tục đăng ký thành lập, tổ chức lại và giải thể doanh nghiệp.",

    # Nhóm: Luật Đất đai & Thuế
    "Khoản 1 Điều 166 Luật Đất đai 2013 quy định người sử dụng đất có quyền được cấp Giấy chứng nhận quyền sử dụng đất, quyền sở hữu nhà ở và tài sản khác gắn liền với đất.",
    "Theo Điều 5 Luật Thuế giá trị gia tăng 2008 (sửa đổi), hàng hóa, dịch vụ không chịu thuế GTGT bao gồm sản phẩm trồng trọt, chăn nuôi, thủy sản chưa qua chế biến.",

    # Nhóm: Văn bản sửa đổi (kiểm tra dẫn chiếu chéo)
    "Luật số 35/2018/QH14 sửa đổi, bổ sung một số điều của 37 Luật liên quan đến quy hoạch; có hiệu lực thi hành từ ngày 01 tháng 01 năm 2019.",
    "Nghị định 47/2021/NĐ-CP ngày 01/4/2021 quy định chi tiết một số điều của Luật Doanh nghiệp 2020, bãi bỏ một số điều của Nghị định 01/2021/NĐ-CP.",

    # Nhóm: Lĩnh vực khác (kiểm tra lọc không liên quan)
    "Quyết định 4689/QĐ-BYT ngày 06/11/2020 ban hành hướng dẫn chẩn đoán và điều trị bệnh COVID-19 do chủng vi-rút Corona mới (SARS-CoV-2) gây ra.",
]
db.add_documents(mock_data)

## Bước 5 — Các Kịch Bản Kiểm Thử

Ba kịch bản kiểm tra toàn bộ pipeline:
- **Kịch bản A** — truy vấn trực tiếp: vector DB trả về tài liệu đúng ngay lần đầu
- **Kịch bản B** — ảo giác: câu trả lời vượt quá nội dung ngữ cảnh, kích hoạt fallback
- **Kịch bản C** — mơ hồ: không tìm được tài liệu liên quan sau nhiều lần thử, trả về phản hồi "không có thông tin"

In [ ]:
test_scenarios = [
    {
        "name": "KỊCH BẢN A: TRUY VẤN TRỰC TIẾP",
        "query": "Số giờ làm thêm tối đa trong một ngày theo Bộ luật Lao động là bao nhiêu?"
    },
    {
        "name": "KỊCH BẢN B: PHÁT HIỆN ẢO GIÁC",
        "query": "Công ty cổ phần cần tối thiểu bao nhiêu cổ đông và vốn điều lệ tối thiểu là bao nhiêu?"
    },
    {
        "name": "KỊCH BẢN C: VIẾT LẠI TRUY VẤN",
        "query": "Tôi muốn biết về cái quy định liên quan đến đất đai ở Việt Nam cho người dân?"
    }
]

print("==============================================")
print("    BẮT ĐẦU DEMO AGENTIC RAG PHÁP LÝ VIỆT NAM")
print("==============================================\n")

for scenario in test_scenarios:
    print(f"\n--- Chạy {scenario['name']} ---")
    print(f"Câu hỏi: '{scenario['query']}'")

    final_output = agentic_rag_loop(db, scenario['query'])

    print(f"\n[Kết quả cuối]: {final_output}")
    print("\n" + "=" * 50)